<a href="https://colab.research.google.com/github/sadhika-tech/deep-learning-lab/blob/main/ResNet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import numpy as np
import time
import gc

from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

In [2]:

tf.keras.backend.clear_session()
gc.collect()

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [3]:
gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU available:", gpus)

    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled.")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found. Go to:")
    print("Runtime -> Change runtime type -> GPU")

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU memory growth enabled.


In [4]:
from tensorflow.keras.datasets import cifar10

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print("Training data shape:", x_train.shape)
print("Testing data shape:", x_test.shape)
print("Training labels shape:", y_train.shape)
print("Testing labels shape:", y_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2561s 15us/step
Training data shape: (50000, 32, 32, 3)
Testing data shape: (10000, 32, 32, 3)
Training labels shape: (50000, 1)
Testing labels shape: (10000, 1)


In [5]:
class_names = [
    "Airplane",
    "Automobile",
    "Bird",
    "Cat",
    "Deer",
    "Dog",
    "Frog",
    "Horse",
    "Ship",
    "Truck"
]


In [6]:
BATCH_SIZE = 8

train_ds = tf.data.Dataset.from_tensor_slices(
    (x_train, y_train)
)

train_ds = train_ds.shuffle(
    buffer_size=10000
)

train_ds = train_ds.batch(
    BATCH_SIZE
)

train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)
test_ds = tf.data.Dataset.from_tensor_slices(
    (x_test, y_test)
)

test_ds = test_ds.batch(
    BATCH_SIZE
)

test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)


print("Batch size:", BATCH_SIZE)
print("Data pipeline created successfully.")

Batch size: 8
Data pipeline created successfully.


In [8]:
def build_resnet50_transfer(num_classes=10):
    inputs = layers.Input(
        shape=(32, 32, 3)
    )
    x = layers.Resizing(
        224,
        224
    )(inputs)
    x = preprocess_input(x)
    base_model = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = False
    x = base_model(
        x,
        training=False
    )
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        256,
        activation="relu"
    )(x)

    x = layers.Dropout(
        0.5
    )(x)

    outputs = layers.Dense(
        num_classes,
        activation="softmax"
    )(x)

    model = models.Model(
        inputs=inputs,
        outputs=outputs,
        name="ResNet50_Transfer"
    )

    return model


In [9]:
model = build_resnet50_transfer(
    num_classes=10
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


In [10]:
model.summary()

Model: "ResNet50_Transfer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resizing (Resizing) │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 224, 224)  │          0 │ resizing[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 224, 224)  │          0 │ resizing[0][0]    │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 224, 224)  │          0 │ resizing[0][0]    │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 224, 224,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 224, 224,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add[0][0]         │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    524,544 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 10)        │      2,570 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,114,826 (91.99 MB)

 Trainable params: 527,114 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [11]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=[
        "accuracy"
    ]
)

In [12]:
EPOCHS = 10

print("\nStarting ResNet50 training...")
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)

start_time = time.time()

history = model.fit(
    train_ds,

    epochs=EPOCHS,

    validation_data=test_ds,

    verbose=1
)

end_time = time.time()


Starting ResNet50 training...
Batch size: 8
Epochs: 10
Epoch 1/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 196s 29ms/step - accuracy: 0.8237 - loss: 0.5238 - val_accuracy: 0.8901 - val_loss: 0.3212
Epoch 2/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 184s 29ms/step - accuracy: 0.8854 - loss: 0.3353 - val_accuracy: 0.9020 - val_loss: 0.2868
Epoch 3/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 184s 29ms/step - accuracy: 0.8994 - loss: 0.2895 - val_accuracy: 0.9049 - val_loss: 0.2799
Epoch 4/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 184s 29ms/step - accuracy: 0.9116 - loss: 0.2554 - val_accuracy: 0.9065 - val_loss: 0.2763
Epoch 5/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 184s 29ms/step - accuracy: 0.9184 - loss: 0.2339 - val_accuracy: 0.9132 - val_loss: 0.2622
Epoch 6/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 185s 30ms/step - accuracy: 0.9237 - loss: 0.2147 - val_accuracy: 0.9115 - val_loss: 0.2704
Epoch 7/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 184s 29ms/step - accuracy: 0.9313 - loss: 0.1977 - val_accuracy: 0.9134 - val_loss: 0.2576
Epoch 8/10
62

In [18]:
training_time = end_time - start_time
print(
    f"Training Time: {training_time} seconds"
)

print(
    f"Training Time: {training_time} minutes"
)

Training Time: 1854.3675141334534 seconds
Training Time: 1854.3675141334534 minutes


In [15]:
print("\nEvaluating model...")

test_loss, test_accuracy = model.evaluate(
    test_ds,
    verbose=1
)

print(
    f"Test Loss: {test_loss}"
)

print(
    f"Test Accuracy: {test_accuracy}"
)

print(
    f"Test Accuracy: {test_accuracy}"
)



Evaluating model...
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 32s 25ms/step - accuracy: 0.9165 - loss: 0.2555
Test Loss: 0.25551745295524597
Test Accuracy: 0.9164999723434448
Test Accuracy: 0.9164999723434448


In [16]:
print("\nGenerating predictions...")

y_pred_prob = model.predict(
    test_ds,
    verbose=1
)

y_pred = np.argmax(
    y_pred_prob,
    axis=1
)

y_true = y_test.flatten()


Generating predictions...
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 34s 24ms/step


In [20]:
print(
    f"Training Accuracy: "
    f"{history.history['accuracy'][-1]}"
)

print(
    f"Validation Accuracy: "
    f"{history.history['val_accuracy'][-1]}"
)

print(
    f"Test Accuracy: "
    f"{test_accuracy}"
)

print(
    f"Training Time: {training_time}"
)

Training Accuracy: 0.9437599778175354
Validation Accuracy: 0.9164999723434448
Test Accuracy: 0.9164999723434448
Training Time: 1854.3675141334534
